# Librerias

In [1]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import re
import causalidad as cs

Initializing causalidad package...
causalidad package initialized successfully.


# Lectura de datos

In [2]:
df_icare = pd.read_excel(r"C:\Users\afpue\OneDrive\Documentos\GitHub\causalidad\Codigo\df_icare.xlsx")

# Funciones

## Calcular el ATE

In [3]:
cols = ['actfreq_sq001', 'recgov_sq001']

df_icare[cols] = df_icare[cols].apply(lambda x: (x == 1).astype(int))
resultado = cs.calcular_ate(
    df=df_icare,
    resultado='actfreq_sq001',
    tratamiento='recgov_sq001',
    covariables=['sex', 'age', 'edu']
)

if resultado is not None:
    print(f"ATE (diferencia de medias): {resultado['ate_diff']:.4f}")
    print(f"ATE (regresión ajustada):   {resultado['ate_reg']:.4f} (p-valor: {resultado['p_valor_reg']:.4f})")
    print(f"N tratados: {resultado['n_tratados']}, N controles: {resultado['n_controles']}")

ATE (diferencia de medias): 0.4244
ATE (regresión ajustada):   0.4119 (p-valor: 0.0000)
N tratados: 1645, N controles: 18


## Balancear

In [15]:
# Para matching
df_match = cs.balancear_propensity(df_icare, 'recgov_sq001', 'actfreq_sq001', 
                                     ['sex','age','edu'], metodo='matched', replacement=False)

ate_match = cs.calcular_ate(df_match, 'actfreq_sq001', 'recgov_sq001', covariables=['sex','age','edu'])

if ate_match is not None:
    print(f"ATE (diferencia de medias): {ate_match['ate_diff']:.4f}")
    print(f"ATE (regresión ajustada):   {ate_match['ate_reg']:.4f} (p-valor: {ate_match['p_valor_reg']:.4f})")
    print(f"N tratados: {ate_match['n_tratados']}, N controles: {ate_match['n_controles']}")

# Para subclassification
df_sub = cs.balancear_propensity(df_icare, 'recgov_sq001', 'actfreq_sq001', 
                                   ['sex','age','edu'], metodo='subclassification', n_subclases=5)

ate_sub = cs.calcular_ate(df_sub, 'actfreq_sq001', 'recgov_sq001', covariables=['sex','age','edu'])

if ate_sub is not None:
    print(f"ATE (diferencia de medias): {ate_sub['ate_diff']:.4f}")
    print(f"ATE (regresión ajustada):   {ate_sub['ate_reg']:.4f} (p-valor: {ate_sub['p_valor_reg']:.4f})")
    print(f"N tratados: {ate_sub['n_tratados']}, N controles: {ate_sub['n_controles']}")

ATE (diferencia de medias): 0.4444
ATE (regresión ajustada):   0.4253 (p-valor: 0.0062)
N tratados: 18, N controles: 18
ATE (diferencia de medias): 0.4244
ATE (regresión ajustada):   0.4119 (p-valor: 0.0000)
N tratados: 1645, N controles: 18
